# 32 — Information Delay Analysis

How quickly does the Kalshi market react to score changes vs our boxscore data? If the market moves before our data updates, our "edge" is illusory.

In [ ]:
import sys
sys.path.insert(0, '/Users/chriswang/Desktop/prediction-market-exploration/nba-edge')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from nba_edge.data.s3_reader import discover_trade_dates, load_trades, load_boxscores_for_game, discover_game_ids
from nba_edge.data.ticker_parser import parse_ticker, is_game_winner
from nba_edge.data.game_alignment import match_ticker_to_game
from nba_edge.models.analytical import AnalyticalWinProb, parse_game_clock

In [ ]:
# Load one well-traded game
dates = discover_trade_dates(min_date='2026-04-21')
df = load_trades(dates)
game_tickers = [t for t in df['market_ticker'].unique().to_list() if is_game_winner(t)]

# Pick the most traded
ticker_volumes = [(t, df.filter(pl.col('market_ticker') == t).height) for t in game_tickers]
ticker_volumes.sort(key=lambda x: -x[1])
chosen = ticker_volumes[0][0]
parsed = parse_ticker(chosen)

available = discover_game_ids(parsed.game_date)
matched = match_ticker_to_game(chosen, available)
snapshots = load_boxscores_for_game(matched, parsed.game_date)
ticker_trades = df.filter(pl.col('market_ticker') == chosen).sort('t_receipt_ns')

print(f'Game: {chosen}')
print(f'Snapshots: {len(snapshots)}, Trades: {len(ticker_trades)}')

In [ ]:
# Detect score change events from boxscore snapshots
score_changes = []  # (t_receipt, old_diff, new_diff, delta)
for i in range(1, len(snapshots)):
    prev_diff = snapshots[i-1]['home_score'] - snapshots[i-1]['away_score']
    curr_diff = snapshots[i]['home_score'] - snapshots[i]['away_score']
    if curr_diff != prev_diff:
        score_changes.append({
            't_change_detected': snapshots[i]['t_receipt'],
            't_prev_snapshot': snapshots[i-1]['t_receipt'],
            'old_diff': prev_diff,
            'new_diff': curr_diff,
            'delta': curr_diff - prev_diff,
        })

print(f'Score changes detected: {len(score_changes)}')
print(f'Average detection delay (between snapshots): {np.mean([s["t_change_detected"] - s["t_prev_snapshot"] for s in score_changes]):.1f}s')

In [ ]:
# For each score change, find the closest trade BEFORE and AFTER
# Measure: how much did the trade price move around the score change?
trade_times = ticker_trades['t_receipt_ns'].to_numpy() / 1e9
trade_prices = ticker_trades['price'].to_numpy()

reactions = []
for sc in score_changes:
    t_detected = sc['t_change_detected']
    
    # Trades in window [-30s, +30s] around detection
    mask_before = (trade_times > t_detected - 30) & (trade_times < t_detected)
    mask_after = (trade_times > t_detected) & (trade_times < t_detected + 30)
    
    if mask_before.any() and mask_after.any():
        price_before = trade_prices[mask_before][-1]  # last trade before
        price_after = trade_prices[mask_after][0]      # first trade after
        
        # When did price actually move? Find first trade that differs significantly
        first_move_idx = np.where(mask_after)[0][0]
        for j in range(first_move_idx, min(first_move_idx + 50, len(trade_prices))):
            if abs(trade_prices[j] - price_before) >= 2:  # 2c move
                market_reaction_time = trade_times[j] - t_detected
                break
        else:
            market_reaction_time = None
        
        reactions.append({
            'delta': sc['delta'],
            'price_before': price_before,
            'price_after': price_after,
            'price_change': price_after - price_before,
            'market_reaction_s': market_reaction_time,
            'detection_delay_s': t_detected - sc['t_prev_snapshot'],
        })

print(f'Score changes with surrounding trades: {len(reactions)}')
if reactions:
    reaction_times = [r['market_reaction_s'] for r in reactions if r['market_reaction_s'] is not None]
    if reaction_times:
        print(f'Market reaction time after our detection:')
        print(f'  mean:   {np.mean(reaction_times):.1f}s')
        print(f'  median: {np.median(reaction_times):.1f}s')
        print(f'  min:    {np.min(reaction_times):.1f}s')
        print(f'  max:    {np.max(reaction_times):.1f}s')

In [ ]:
# Key question: does the market move BEFORE we detect the score change?
# If market_reaction_s < 0, the market knew before us

# Look at price changes in the 30s window BEFORE our detection
early_movers = 0
for sc in score_changes:
    t_detected = sc['t_change_detected']
    # Direction the price should move (positive delta = home scoring = price up for home YES)
    expected_dir = np.sign(sc['delta'])
    
    # Check trades 5-20s BEFORE our detection
    mask_pre = (trade_times > t_detected - 20) & (trade_times < t_detected - 2)
    if mask_pre.sum() >= 2:
        pre_prices = trade_prices[mask_pre]
        pre_move = pre_prices[-1] - pre_prices[0]
        # Did the market move in the expected direction before we even knew?
        if np.sign(pre_move) == expected_dir and abs(pre_move) >= 2:
            early_movers += 1

print(f'Score changes where market moved first (before our data): {early_movers}/{len(score_changes)}')
print(f'Rate: {early_movers/len(score_changes)*100:.1f}%')
print()
print('If this rate is high, the market has faster information than our')
print('boxscore polling and our "edge" is actually information lag.')

In [ ]:
# Visualize a few score change events
fig, axes = plt.subplots(min(4, len(score_changes)), 1, figsize=(14, 3*min(4, len(score_changes))))
if not hasattr(axes, '__len__'):
    axes = [axes]

for idx, sc in enumerate(score_changes[:4]):
    t_detected = sc['t_change_detected']
    window = 60  # seconds around event
    
    mask = (trade_times > t_detected - window) & (trade_times < t_detected + window)
    if not mask.any():
        continue
    
    t_rel = trade_times[mask] - t_detected  # seconds relative to detection
    prices = trade_prices[mask]
    
    axes[idx].plot(t_rel, prices, 'o-', markersize=3, linewidth=0.8)
    axes[idx].axvline(0, color='red', linestyle='--', alpha=0.7, label='Score change detected')
    axes[idx].set_ylabel('Price (c)')
    axes[idx].set_title(f'Score Δ={sc["delta"]:+d} (diff: {sc["old_diff"]}→{sc["new_diff"]})')
    axes[idx].legend(fontsize=8)

axes[-1].set_xlabel('Seconds relative to score change detection')
plt.tight_layout()
plt.show()

In [ ]:
# Summary: information advantage/disadvantage
detection_delays = [sc['t_change_detected'] - sc['t_prev_snapshot'] for sc in score_changes]

print('=== INFORMATION DELAY SUMMARY ===')
print(f'Boxscore poll interval: {np.mean(detection_delays):.1f}s avg')
print(f'  → We learn about scores {np.mean(detection_delays):.0f}s after they happen (worst case)')
print(f'  → Actual delay is somewhere between 0 and {np.mean(detection_delays):.0f}s (uniform)')
print()
if reactions and reaction_times:
    print(f'Market reaction after our detection: {np.median(reaction_times):.1f}s median')
    if np.median(reaction_times) > 0:
        print('  → Market reacts AFTER us — we may have an information edge!')
    else:
        print('  → Market reacts BEFORE us — no information advantage')
print()
print(f'Market front-running rate: {early_movers/len(score_changes)*100:.0f}%')
print('(High = market is faster than our data; edge is likely illusory)')